# Qwen 7B Baseline Endpoint Packaging Test

This notebook is for deployment isolation testing only.

It does not train and does not merge any adapter. It only:
- loads baseline Qwen 7B weights from Hugging Face
- saves and optionally pushes those same weights to your own model repo
- supports endpoint deployment from your owned repo for fair endpoint-vs-endpoint evaluation

Use this to test serving/endpoint effects independent of fine-tuning.

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes huggingface_hub sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random
import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged into Hugging Face as:', whoami()['name'])
else:
    raise RuntimeError('HF_TOKEN is missing in Colab secrets. Add it, then rerun this cell.')

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    base_model: str = 'Qwen/Qwen2.5-7B-Instruct'

    # Baseline packaging only: no adapter merge in this notebook.
    packaged_output_dir: str = '/content/drive/MyDrive/triples/train_dev_val/qwen7b_baseline_packaged'
    packaged_repo_id: str = 'dizza01/qwen7b-baseline-packaged'

    torch_dtype: str = 'float16'
    max_shard_size: str = '5GB'
    push_packaged: bool = True

cfg = Config()
cfg

In [ ]:
from transformers import AutoTokenizer

print('Base model:', cfg.base_model)
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded from base model')

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM

dtype = torch.float16 if cfg.torch_dtype == 'float16' else torch.bfloat16

print('Loading baseline model from:', cfg.base_model)
packaged_model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    torch_dtype=dtype,
    device_map='auto',
    trust_remote_code=True,
)

os.makedirs(cfg.packaged_output_dir, exist_ok=True)
packaged_model.save_pretrained(
    cfg.packaged_output_dir,
    safe_serialization=True,
    max_shard_size=cfg.max_shard_size,
)
tokenizer.save_pretrained(cfg.packaged_output_dir)

print('Baseline model packaged to:', cfg.packaged_output_dir)

In [ ]:
from transformers import AutoModelForCausalLM, pipeline

print('Loading packaged baseline model for a local smoke test...')
smoke_model = AutoModelForCausalLM.from_pretrained(
    cfg.packaged_output_dir,
    torch_dtype=dtype,
    device_map='auto',
    trust_remote_code=True,
)

gen = pipeline(
    'text-generation',
    model=smoke_model,
    tokenizer=tokenizer,
    do_sample=False,
    temperature=0.0,
    return_full_text=False,
    max_new_tokens=180,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
 )

prompt = (
    'You are a helpful research assistant.\n\n'
    'Retrieved knowledge base context:\n\n'
    'Prenatal alcohol exposure is associated with poorer early education outcomes.\n\n'
    '---\n\n'
    'Researcher question: What is the relationship between prenatal alcohol exposure and early education outcomes?'
)

result = gen(prompt)
print(result[0]['generated_text'])

In [ ]:
from huggingface_hub import HfApi, whoami

if not cfg.push_packaged:
    print('Skipping push. Set cfg.push_packaged=True to upload packaged baseline model.')
else:
    print('Pushing as:', whoami()['name'])
    print('Target repo:', cfg.packaged_repo_id)

    api = HfApi()
    api.create_repo(repo_id=cfg.packaged_repo_id, repo_type='model', exist_ok=True)

    packaged_model.push_to_hub(cfg.packaged_repo_id, max_shard_size=cfg.max_shard_size)
    tokenizer.push_to_hub(cfg.packaged_repo_id)

    print('Packaged baseline model pushed to: https://huggingface.co/' + cfg.packaged_repo_id)

## Endpoint Deployment Notes

After pushing, deploy an endpoint from the packaged baseline repo and keep inference settings aligned with your trained endpoint run for fair comparison:
- same endpoint task type
- same generation limits (max_new_tokens, temperature, stop behavior)
- same request format (text_generation vs chat_completions)

Then run faithfulness eval against this baseline endpoint:

```bash
../../.venv/bin/python eval/run_faithfulness_eval_updated.py \
  --triples eval/evaluation_datasets/triples/train_dev_val/sft_test.jsonl \
  --answer-model Qwen/Qwen2.5-7B-Instruct \
  --answer-api-mode hf_endpoint \
  --answer-endpoint-url https://YOUR-BASELINE-ENDPOINT \
  --answer-endpoint-mode text_generation \
  --retrieval-mode default \
  --dense-top-k 5 \
  --answer-max-tokens 900 \
  --judge-max-tokens 700 \
  --external-judge-model meta-llama/Llama-3.1-70B-Instruct \
  --qwen-judge-model Qwen/Qwen2.5-72B-Instruct \
  --judge-retries 2 \
  --judge-retry-delay 0.5 \
  --run-name ab_qwen7b_baseline_endpoint_sfttest
```